# Localizing AutoAttack noise: the statistic that finally works

[`../06-Attack-Benchmark/`](../06-Attack-Benchmark/) predicted the high-frequency energy gate would
fail on attacks that are not broadband. [`../07-Attack-Repair/`](../07-Attack-Repair/) measured that
failure and reported a **negative result**: IoU 0.05–0.16 at ~15% false positives, because
**median band energy varies ~6× between clean photos while a bounded attack lifts local energy only
~2.5× — the confound is bigger than the signal.**

This notebook fixes that, on **real AutoAttack perturbations** and **my own Kaggle dataset**.

### The idea in one sentence

The problem was never the filter bank — it was that **energy is not contrast-invariant**. So score
the **shape** of the local spectrum instead of its scale: the ratio of second-difference to
first-difference energy. Contrast cancels in the ratio, so a dark sky and a bright textured wall
give the same number, while additive noise moves it.

For white (L∞) noise that ratio is **exactly 3.0**; natural content sits well below it.

### Two design points that came out of measurement, not intuition

1. **The score must be two-sided.** L∞ noise makes the local spectrum *flatter* than natural;
   smooth L2 noise (C&W/DeepFool-like) makes it *steeper*. A one-sided detector fitted on L∞
   scores smooth attacks at AUC ≈ 0.34 — that is not failure, it is **inversion**. Scoring the
   `|deviation|` from the image's own natural value catches both directions.
2. **The learned fusion rediscovers the same thesis.** Given five features it puts large
   opposite-sign weights on SRM energy and plain band energy — i.e. it constructs a *ratio*, which
   is a scale-invariant spectral-shape statistic. The same idea as point 1, arrived at
   independently.

### What this notebook measures

Real `torchattacks` AutoAttack components (APGD-CE, APGD-T, FAB-T, Square) crafted against a
ResNet-50 and confined to random non-rectangular regions on 5 random images from my
[obstacle-detection dataset](https://www.kaggle.com/datasets/abtinzandi/obstacle-detection-dataset),
plus RMS-matched smooth-L2 and low-frequency controls that AutoAttack cannot produce.

Reported under `07`'s methodology rules, which exist because breaking them produced confidently
wrong conclusions before: **per-pixel ROC-AUC** as the threshold-free primary metric, **IoU at a
false-positive rate calibrated on clean images only**, and **leave-one-image-out** validation with
the fusion **fitted on APGD-CE alone** and tested on every other attack — so no number comes from
training on its own test attack.

**Environment: Kaggle, accelerator GPU T4 ×2, Internet On.** Both T4s are used for the attack (the
expensive iterative part); the detector is a handful of convolutions and runs batched on one.
Expect roughly 10–15 minutes end to end.

## 0. Setup

In [ ]:
!pip install -q torchattacks kagglehub ultralytics

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import transforms, models
from PIL import Image, ImageDraw
import glob, os, time, threading, random

SIZE    = 512        # repo-standard working resolution
EPS     = 8/255      # the AutoAttack / RobustBench standard Linf budget
SEED    = 0
SIGMA_L = 6.0        # radius of the sliding window every local statistic uses

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
NDEV = len(DEVS)
DEV  = DEVS[0]
print("devices:", DEVS)

# --- generic utilities, used by both the attack synthesis and the detector ---
def gauss1d(sigma):
    k  = int(6*sigma) | 1
    ax = torch.arange(k) - k//2
    g  = torch.exp(-(ax.float()**2)/(2*sigma**2))
    return g/g.sum()

def blur_sep(x, sigma):
    # the 2D Gaussian is an outer product, so two 1D convs are exact and ~25x cheaper
    g = gauss1d(sigma).to(x.device, x.dtype)
    k = g.numel(); p = k//2; C = x.shape[1]
    x = F.conv2d(x, g.view(1,1,1,k).repeat(C,1,1,1), padding=(0,p), groups=C)
    x = F.conv2d(x, g.view(1,1,k,1).repeat(C,1,1,1), padding=(p,0), groups=C)
    return x

gray  = lambda x: x.mean(1, keepdim=True)
local = lambda x, s=SIGMA_L: blur_sep(x, s)

## 1. Five random images from my Kaggle dataset

Two ways in, tried in order:

1. **`+ Add Input` in the Kaggle sidebar** → the dataset appears under `/kaggle/input/...`. Fastest,
   no download, works with Internet Off.
2. **`kagglehub.dataset_download(...)`** → downloads it. Needs Internet On.

A note on the snippet Kaggle shows for this dataset: `KaggleDatasetAdapter.PANDAS` loads one
**tabular file** into a DataFrame. This is an image dataset, so what we want is the files on disk —
`dataset_download` returns the folder and we glob the images out of it. If the dataset also ships a
labels CSV, the PANDAS adapter is the right tool for *that file*, not for the pictures.

In [ ]:
DATASET  = "abtinzandi/obstacle-detection-dataset"
N_IMAGES = 5
EXT      = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def find_images():
    for r in sorted(glob.glob("/kaggle/input/*")):            # 1. mounted via + Add Input
        f = [p for p in glob.glob(os.path.join(r, "**", "*"), recursive=True)
             if p.lower().endswith(EXT)]
        if f:
            print(f"using mounted input: {r}  ({len(f)} images)")
            return f
    import kagglehub                                          # 2. download
    p = kagglehub.dataset_download(DATASET)
    f = [q for q in glob.glob(os.path.join(p, "**", "*"), recursive=True)
         if q.lower().endswith(EXT)]
    print(f"downloaded to: {p}  ({len(f)} images)")
    return f

files = find_images()
assert files, "no images found - add the dataset via + Add Input, or enable Internet"

rng    = np.random.default_rng(SEED)
picked = [files[i] for i in rng.choice(len(files), size=min(N_IMAGES, len(files)), replace=False)]

tt  = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
raw = torch.cat([tt(Image.open(p).convert("RGB")).unsqueeze(0) for p in picked]).to(DEV)
N   = raw.shape[0]
print(f"\n{N} random images @ {SIZE}x{SIZE}:")
for p in picked: print("  ", os.path.basename(p))

fig, ax = plt.subplots(1, N, figsize=(3.2*N, 3.4))
for i, a in enumerate(np.atleast_1d(ax)):
    a.imshow(raw[i].permute(1,2,0).cpu()); a.set_title(f"image {i}"); a.axis("off")
plt.suptitle("the 5 randomly drawn images"); plt.tight_layout(); plt.show()

## 2. Random attack regions — deliberately not rectangles

One random region per image, drawn from blob / ring / scribble / wedge. Nothing is tile-aligned,
because the detector below has no tiles: every statistic is per-pixel, so the shape of the attacked
area is unconstrained.

In [ ]:
def shape_mask(kind, seed=0, size=SIZE):
    g  = np.random.default_rng(seed)
    im = Image.new("L", (size, size), 0); d = ImageDraw.Draw(im)
    if kind == "blob":
        cx, cy = g.uniform(0.3, 0.7, 2)*size
        a = np.sort(g.uniform(0, 2*np.pi, 9)); r = g.uniform(0.14, 0.32, 9)*size
        d.polygon([(float(cx+ri*np.cos(ai)), float(cy+ri*np.sin(ai))) for ai, ri in zip(a, r)], fill=1)
    elif kind == "ring":
        c = g.uniform(0.35, 0.65, 2)*size; R = g.uniform(0.22, 0.34)*size
        d.ellipse([c[0]-R, c[1]-R, c[0]+R, c[1]+R], fill=1)
        d.ellipse([c[0]-R/2, c[1]-R/2, c[0]+R/2, c[1]+R/2], fill=0)
    elif kind == "scribble":
        y0  = g.uniform(0.25, 0.75)*size
        pts = [(0.12*size, y0)] + [(0.12*size + i*0.12*size,
                y0 + 0.18*size*np.sin(i*g.uniform(0.9, 1.6))) for i in range(1, 7)]
        d.line([(float(a), float(b)) for a, b in pts], fill=1, width=int(0.05*size), joint="curve")
    else:                                                     # wedge
        c = g.uniform(0.3, 0.7, 2)*size; R = g.uniform(0.3, 0.45)*size; s = g.uniform(0, 360)
        d.pieslice([c[0]-R, c[1]-R, c[0]+R, c[1]+R], s, s + g.uniform(70, 140), fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

KINDS = ["blob", "ring", "scribble", "wedge"]
masks = torch.cat([shape_mask(KINDS[i % len(KINDS)], seed=100+i) for i in range(N)]).to(DEV)
print("region coverage per image (% of pixels):", [f"{m.mean().item()*100:.1f}" for m in masks])

fig, ax = plt.subplots(1, N, figsize=(3.2*N, 3.4))
for i, a in enumerate(np.atleast_1d(ax)):
    a.imshow(masks[i,0].cpu(), cmap="gray"); a.set_title(KINDS[i % len(KINDS)]); a.axis("off")
plt.suptitle("attack regions (ground truth) - arbitrary shapes"); plt.tight_layout(); plt.show()

## 3. The victim model and real AutoAttack perturbations

AutoAttack needs gradients through a classifier, so the perturbations are crafted against a
pretrained **ResNet-50**, with labels taken from the model's own clean predictions (the standard
convention for unlabelled data). §10 then measures what that same perturbation does to a **YOLO
obstacle detector** — the task this dataset is actually for.

The four AutoAttack components are run **separately** rather than as the ensemble, because "which
component does the detector catch?" is the informative question; the ensemble would only report
their union. Each is a real `torchattacks` implementation.

**One honest caveat.** AutoAttack optimizes over the whole image and has no masked variant, so δ is
computed on the full image and then **confined to the region afterwards** (`x + m·δ`). The noise
*statistics* the detector sees are genuine AutoAttack output, which is what is under test, but the
confined perturbation is no longer optimal for that region — so the printed fooling rate is a lower
bound. A hand-rolled **masked PGD**, which applies the mask at every step, is included as the
properly-confined reference.

Both T4s are used: the batch is split in half and each half attacked on its own GPU in a thread.
The attack is iterative and slow, so this is where parallelism actually pays — unlike the detector,
which `07` showed needs nothing beyond batching.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

class Wrapped(nn.Module):
    # torchattacks expects a model that takes [0,1] images, so fold normalization inside
    def __init__(self, net, dev):
        super().__init__()
        self.net = net.eval().to(dev)
        self.register_buffer("m", MEAN.to(dev)); self.register_buffer("s", STD.to(dev))
    def forward(self, x):
        return self.net((x - self.m)/self.s)

_w   = models.ResNet50_Weights.IMAGENET1K_V2
nets = {d: Wrapped(models.resnet50(weights=_w), d) for d in DEVS}
for n in nets.values():
    for p in n.parameters(): p.requires_grad_(False)

with torch.no_grad():
    labels = nets[DEV](raw).argmax(1)
print("clean ResNet-50 predictions (used as attack labels):", labels.tolist())

In [ ]:
import torchattacks

def build(name, model):
    # torchattacks kwargs shift between versions, so an unavailable attack is skipped
    # rather than killing the run
    try:
        if name == "APGD-CE":
            return torchattacks.APGD(model, norm="Linf", eps=EPS, steps=50, n_restarts=1, loss="ce")
        if name == "APGD-T":
            return torchattacks.APGDT(model, norm="Linf", eps=EPS, steps=50,
                                      n_restarts=1, n_classes=1000)
        if name == "FAB-T":
            return torchattacks.FAB(model, norm="Linf", eps=EPS, steps=50,
                                    n_restarts=1, n_classes=1000, multi_targeted=True)
        if name == "Square":
            return torchattacks.Square(model, norm="Linf", eps=EPS, n_queries=2000, n_restarts=1)
        if name == "AutoAttack":
            return torchattacks.AutoAttack(model, norm="Linf", eps=EPS,
                                           version="standard", n_classes=1000)
    except Exception as e:
        print(f"  [skip] {name}: {type(e).__name__}: {e}")
    return None

AA_NAMES = ["APGD-CE", "APGD-T", "FAB-T", "Square"]   # append "AutoAttack" for the full ensemble

def run_split(name):
    # attack the batch across all available GPUs, one thread per device
    chunks = np.array_split(np.arange(N), NDEV)
    out    = [None]*NDEV
    def work(gi):
        idx = chunks[gi]
        if len(idx) == 0: return
        d = DEVS[gi]
        if d.startswith("cuda"): torch.cuda.set_device(gi)
        atk = build(name, nets[d])
        if atk is None: return
        out[gi] = atk(raw[idx].to(d), labels[idx].to(d)).to(DEV)
    ts = [threading.Thread(target=work, args=(g,)) for g in range(NDEV)]
    for t in ts: t.start()
    for t in ts: t.join()
    got = [o for o in out if o is not None]
    if len(got) != sum(1 for c in chunks if len(c)): return None
    return torch.cat(got)

adv_full = {}
for nm in AA_NAMES:
    t0 = time.perf_counter()
    a  = run_split(nm)
    if a is not None:
        adv_full[nm] = a
        print(f"{nm:11s} done in {time.perf_counter()-t0:6.1f}s   "
              f"max|delta| = {(a-raw).abs().max().item()*255:.2f}/255")
assert adv_full, "no attack ran - check the torchattacks version"

In [ ]:
def rms_in(v, m):
    mm = m.expand_as(v) > 0.5
    return (v[mm]**2).mean().sqrt()

def masked_pgd(x, y, m, steps=50, alpha=EPS/8):
    # properly-confined reference: the mask is applied at EVERY step, so the optimizer
    # only ever spends budget inside the region
    net = nets[DEV]
    d   = (torch.rand_like(x)*2 - 1)*EPS*m
    for _ in range(steps):
        d.requires_grad_(True)
        g, = torch.autograd.grad(F.cross_entropy(net((x+d).clamp(0,1)), y), d)
        d  = (d.detach() + alpha*g.sign()*m).clamp(-EPS, EPS)
    return (x + d*m).clamp(0, 1)

def synth(kind, x, m, ref_rms, seed=0):
    # controls AutoAttack cannot produce, RMS-matched inside the mask to the real attack so
    # that nothing wins on raw energy alone (a mistake 07 paid for)
    g = torch.Generator().manual_seed(seed)
    n = torch.randn(x.shape, generator=g).to(x.device)
    d = blur_sep(n, 1.5) if kind == "L2-smooth" else blur_sep(n, 8.0)
    return (x + d/(rms_in(d, m) + 1e-10)*ref_rms*m).clamp(0, 1)

## 4. The detector: five per-pixel features, batched, no tiles

Every feature is a local statistic computed at full resolution. `07` established why there are no
tiles: `conv2d` zero-pads each tile independently, fabricating a black neighbour, which inflated
the coarse bands by 24× on interior tiles.

The feature that matters is **`slope`**. Contrast appears in both the numerator and denominator of
`e2/e1` and cancels, so it measures spectral *shape*. White noise gives exactly 3.0; natural
content sits well below.

In [ ]:
# Kirchner-Vahid 5x5 residual. The SRM / spatial-rich-model family is the literature's strongest
# hand-crafted adversarial-detection feature set (arXiv:1806.09186).
KV = torch.tensor([[-1,  2,  -2,  2, -1],
                   [ 2, -6,   8, -6,  2],
                   [-2,  8, -12,  8, -2],
                   [ 2, -6,   8, -6,  2],
                   [-1,  2,  -2,  2, -1]], dtype=torch.float32)/12.

def _pad_to(t, ref):
    return F.pad(t, (0, ref.shape[-1]-t.shape[-1], 0, ref.shape[-2]-t.shape[-2]), mode="replicate")

def f_energy(x):
    # 07's statistic: log local energy of the finest band. Scale-DEPENDENT, hence its failure.
    return torch.log(local(gray(x - blur_sep(x, 1.0))**2) + 1e-10)

def f_srm(x):
    return torch.log(local(F.conv2d(gray(x), KV.view(1,1,5,5).to(x.device), padding=2)**2) + 1e-10)

def f_slope(x):
    # THE feature: local 2nd-difference energy over 1st-difference energy.
    # Contrast cancels in the ratio. White noise -> 3.0, natural content -> well below.
    g   = gray(x)
    d1x = g[..., :, 1:] - g[..., :, :-1]
    d1y = g[..., 1:, :] - g[..., :-1, :]
    d2x = g[..., :, 2:] - 2*g[..., :, 1:-1] + g[..., :, :-2]
    d2y = g[..., 2:, :] - 2*g[..., 1:-1, :] + g[..., :-2, :]
    e1  = local(_pad_to(d1x, g)**2 + _pad_to(d1y, g)**2)
    e2  = local(_pad_to(d2x, g)**2 + _pad_to(d2y, g)**2)
    return e2/(e1 + 1e-10)

def f_rho1(x):
    # negated lag-1 autocorrelation of the high-pass residual: also a ratio, also scale-free.
    # Natural texture is spatially correlated; injected noise is not.
    r   = gray(x - blur_sep(x, 1.0))
    num = _pad_to(local(r[..., :, 1:]*r[..., :, :-1]), r)
    return -num/(local(r**2) + 1e-10)

def f_slope_coarse(x):
    # the same ratio one octave down - the band where a low-frequency poison should live
    b  = blur_sep(gray(x), 2.0)
    d1 = b[..., :, 2:] - b[..., :, :-2]
    d2 = b[..., :, 4:] - 2*b[..., :, 2:-2] + b[..., :, :-4]
    return local(_pad_to(d2, b)**2)/(local(_pad_to(d1, b)**2) + 1e-10)

FEATS = {"energy (07)": f_energy, "SRM": f_srm, "slope": f_slope,
         "-rho1": f_rho1, "slope-coarse": f_slope_coarse}

def two_sided(f):
    # |robust z| against each image's OWN median/MAD -> [N,H,W].
    # Per-image because cross-image calibration is hopeless (07 measured a 6x spread), and
    # ABSOLUTE because the deviation is signed by attack type: Linf flattens the local spectrum,
    # smooth L2 steepens it. |.| catches both; a one-sided score INVERTS on half the zoo.
    return _z(f).abs()[:, 0]

def one_sided(f):
    # 07's actual gate: upper tail only, `median + k*MAD`. Kept so the baseline in the money
    # figure is the detector 07 really ran, not a two-sided variant it never used.
    return _z(f)[:, 0]

def _z(f):
    v   = f.flatten(1)
    med = v.median(1).values.view(-1,1,1,1)
    mad = (f - med).flatten(1).abs().median(1).values.view(-1,1,1,1) + 1e-10
    return (f - med)/(1.4826*mad)

print(f"{len(FEATS)} features:", list(FEATS))

### The invariance claim, verified on these photos

`07`'s diagnosis was that clean-image energy varies far more *between* photos than an attack shifts
it *within* one. Below: how far each feature's clean median moves across these 5 real images. The
energy features are logs, so their spread is printed in nats as well as the equivalent linear
factor; the ratio features are printed as a plain max/min.

A feature can only carry a global threshold if this spread is small.

In [ ]:
with torch.no_grad():
    print(f"{'feature':14s} {'clean medians across the 5 images':46s} spread")
    print("-"*79)
    for nm, fn in FEATS.items():
        v = [fn(raw[i:i+1]).median().item() for i in range(N)]
        s = " ".join(f"{q:8.3f}" for q in v)
        tag = (f"{max(v)-min(v):.2f} nats = {np.exp(max(v)-min(v)):.1f}x"
               if nm in ("energy (07)", "SRM") else f"{max(v)/min(v):.2f}x")
        print(f"{nm:14s} {s:46s} {tag}")

print("\n"
      "The energy-type features move by a large factor between photos - that spread IS the\n"
      "confound 07 identified. The ratio features barely move, which is the entire point: they\n"
      "measure spectral shape, and shape is what additive noise changes.")

## 5. Assembling the attack zoo

The four AutoAttack components with δ confined to each image's region, the properly-confined masked
PGD, and the two RMS-matched controls. The reference RMS is taken from APGD-CE, so every synthetic
control carries **the same perturbation energy inside the mask** as the real attack — otherwise a
louder attack wins for free, which is exactly how `07` once concluded "low-frequency is harder"
from an attack that was merely 4× weaker.

In [ ]:
REF     = "APGD-CE" if "APGD-CE" in adv_full else list(adv_full)[0]
REF_RMS = rms_in(adv_full[REF] - raw, masks)
print(f"reference RMS inside the region ({REF}): {REF_RMS.item():.4f}")

ZOO = {nm: (raw + (a - raw)*masks).clamp(0, 1) for nm, a in adv_full.items()}
ZOO["masked-PGD"] = masked_pgd(raw, labels, masks)
for k in ["L2-smooth", "low-freq"]:
    ZOO[k] = torch.cat([synth(k, raw[i:i+1], masks[i:i+1], REF_RMS, seed=i) for i in range(N)])

with torch.no_grad():
    print(f"\n{'attack':12s} {'RMS in region':>13s} {'max|d| /255':>12s} {'ResNet-50 fooled':>17s}")
    print("-"*58)
    for nm, a in ZOO.items():
        p = nets[DEV](a).argmax(1)
        print(f"{nm:12s} {rms_in(a-raw, masks).item():13.4f} "
              f"{(a-raw).abs().max().item()*255:12.2f} {f'{(p!=labels).sum().item()}/{N}':>17s}")

print("\n"
      "The fooling rate is a LOWER bound for the four AutoAttack rows: confining an\n"
      "unconstrained delta to a fraction of the pixels throws most of it away. masked-PGD is the\n"
      "fair number for region-confined attack strength. What this notebook measures is detection,\n"
      "not attack success.")

## 6. Money table 1 — per-pixel ROC-AUC, every feature × every attack

AUC is threshold-free, which separates "is the signal there?" from "did I pick a good threshold?".
`07` conflated those and it cost a wrong conclusion. Scores are **pooled across all 5 images** —
one global ranking, the hard version, because a per-image AUC would hide the calibration problem
that broke the energy statistic.

Every feature in the table is scored **two-sided**, so the comparison is like-for-like. The gate
`07` actually deployed was **one-sided** (`median + k·MAD`, upper tail only), so that exact variant
is printed underneath as the reference baseline, and it is the baseline curve in §8 — comparing
against a two-sided version it never ran would be a strawman.

In [ ]:
def auc(score, label):
    # rank-based AUC, on GPU. score/label: 1-D tensors
    s = score.float().flatten(); y = label.flatten() > 0.5
    npos = int(y.sum()); nneg = y.numel() - npos
    if npos == 0 or nneg == 0: return float("nan")
    r = torch.empty_like(s)
    r[s.argsort()] = torch.arange(1, s.numel()+1, device=s.device, dtype=s.dtype)
    return ((r[y].sum() - npos*(npos+1)/2)/(npos*nneg)).item()

GT = masks[:, 0] > 0.5

with torch.no_grad():
    AUCS = {nm: {k: auc(two_sided(fn(a)).flatten(), GT.flatten()) for k, fn in FEATS.items()}
            for nm, a in ZOO.items()}

hdr = f"{'attack':12s}" + "".join(f"{k:>14s}" for k in FEATS)
print(hdr); print("-"*len(hdr))
for nm in ZOO:
    print(f"{nm:12s}" + "".join(f"{AUCS[nm][k]:14.3f}" for k in FEATS))

# the baseline 07 actually ran, for reference: same energy feature, upper tail only
with torch.no_grad():
    ONE = {nm: auc(one_sided(f_energy(a)).flatten(), GT.flatten()) for nm, a in ZOO.items()}
print(f"\n{'(reference) 07 gate = one-sided energy':38s}"
      + "  ".join(f"{nm}:{ONE[nm]:.3f}" for nm in ZOO))

# data-driven version of the claim, so the notebook checks it instead of asserting it
e_mean = np.nanmean([AUCS[nm]["energy (07)"] for nm in ZOO])
s_mean = np.nanmean([AUCS[nm]["slope"]       for nm in ZOO])
print(f"\nmean AUC over the whole zoo:  energy {e_mean:.3f}   vs   slope {s_mean:.3f}")
print("Same images, same attacks, same pooling - the only thing that changed is scoring\n"
      "spectral SHAPE instead of spectral SCALE."
      if s_mean > e_mean else
      "NOTE: slope did NOT beat energy on this draw of images - report that, do not bury it.")

## 7. Money table 2 — the learned fusion, leave-one-image-out, fitted on APGD-CE only

A tiny logistic regression over the five two-sided features — 6 parameters. The protocol is the
strict one:

- **Leave-one-image-out.** Fit on four images, score the fifth. No image is ever scored by a model
  that saw it.
- **Fitted on APGD-CE alone**, then tested on every attack, so the FAB-T, Square, L2 and
  low-frequency columns are all **transfer**, not fitted performance. This is the honest version of
  the "small learned per-tile classifier" that the [`../05-Noise-Gate/`](../05-Noise-Gate/) research
  log has listed as the next step from the start.
- Clean images join the training set as pure negatives.

Five images is a small training set — which is why the protocol is leave-one-out; that is what makes
a number from five images mean anything. Scaling up is the obvious next step and only `N_IMAGES`
changes.

In [ ]:
SUB      = 4      # pixel stride when fitting (speed only)
TRAIN_ON = REF    # fit on APGD-CE; every other attack is transfer

def featmat(x):
    # [1,3,H,W] -> [P,F] matrix of two-sided features
    return torch.cat([two_sided(fn(x)).flatten().unsqueeze(1) for fn in FEATS.values()], 1)

def fit_logreg(X, y, iters=400, lr=0.5):
    mu, sd = X.mean(0), X.std(0) + 1e-8
    Xn = (X - mu)/sd
    w  = torch.zeros(X.shape[1], device=X.device, requires_grad=True)
    b  = torch.zeros(1, device=X.device, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=lr)
    for _ in range(iters):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(Xn@w + b, y).backward()
        opt.step()
    return w.detach(), b.detach(), mu, sd

def apply_logreg(m, X):
    w, b, mu, sd = m
    return ((X - mu)/sd)@w + b

with torch.no_grad():
    Fatt   = {i: featmat(ZOO[TRAIN_ON][i:i+1])[::SUB] for i in range(N)}
    Fclean = {i: featmat(raw[i:i+1])[::SUB]           for i in range(N)}
    ysub   = {i: masks[i].flatten()[::SUB]            for i in range(N)}

LOIO = []
for held in range(N):
    tr = [i for i in range(N) if i != held]
    X  = torch.cat([Fatt[i] for i in tr] + [Fclean[i] for i in tr])
    y  = torch.cat([ysub[i] for i in tr] + [torch.zeros_like(ysub[i]) for i in tr])
    LOIO.append(fit_logreg(X, y))

def fuse_score(z):
    # image i is always scored by the model that did NOT see image i
    with torch.no_grad():
        return torch.cat([apply_logreg(LOIO[i], featmat(z[i:i+1])).view(1, SIZE, SIZE)
                          for i in range(z.shape[0])])

with torch.no_grad():
    FUSED    = {nm: fuse_score(a) for nm, a in ZOO.items()}
    FUSE_AUC = {nm: [auc(FUSED[nm][i].flatten(), GT[i].flatten()) for i in range(N)]
                for nm in ZOO}

print(f"{'attack':12s} {'fusion AUC':>11s}   {'best single feature':>28s}   per-held-out-image")
print("-"*90)
for nm in ZOO:
    b   = max(FEATS, key=lambda k: AUCS[nm][k])
    per = FUSE_AUC[nm]
    tag = "   <- fitted on this" if nm == TRAIN_ON else ""
    print(f"{nm:12s} {np.nanmean(per):11.3f}   {b + f' ({AUCS[nm][b]:.3f})':>28s}   "
          + " ".join(f"{q:.2f}" for q in per) + tag)

W  = torch.stack([m[0] for m in LOIO]).mean(0)
wd = dict(zip(FEATS, W.tolist()))
print("\nmean learned weight (positive = a larger |deviation| means attacked):")
for nm, q in wd.items(): print(f"  {nm:14s} {q:+.3f}")

# self-checking, not asserted: the interesting pattern is opposite signs on the two energy
# features, which is a RATIO - a scale-invariant spectral-shape statistic the model was never
# told to build. If a given draw of images does not reproduce it, say so.
if wd["SRM"]*wd["energy (07)"] < 0:
    print("\n"
          "Opposite signs on SRM and plain band energy. Free to weight five features, the model\n"
          "combined two ENERGY features into a difference of logs - i.e. a RATIO, which is a\n"
          "scale-invariant spectral-shape statistic it was never told to build. The hand-designed\n"
          "'slope' feature and the learned fusion reach the same idea from different directions.")
else:
    print("\n"
          "NOTE: on this draw the two energy weights share a sign, so the fusion did NOT build the\n"
          "ratio seen in earlier runs. Report it as-is - the hand-designed 'slope' feature stands\n"
          "on its own evidence in the table above.")

## 8. Money figure — IoU vs. clean false positives

`07`'s hardest-won rule: **a fixed threshold proves nothing**, because any detector can raise its
IoU by flagging more pixels. So the threshold is chosen **on clean images only**, to hit a target
false-positive rate, then applied unchanged to the attacked images. Better detectors sit **up and
to the left**.

The grey band and dashed line mark where `07` landed: IoU 0.05–0.16 at ~15% false positives.

In [ ]:
FPS            = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
CURVE_ATTACKS  = [k for k in [REF, "Square", "L2-smooth", "low-freq"] if k in ZOO]

def iou_curve(score_of):
    # score_of(batch) -> [N,H,W]. Threshold comes from CLEAN images only; IoU is on attacked.
    clean = score_of(raw).flatten().float()
    att   = {nm: score_of(ZOO[nm]) for nm in CURVE_ATTACKS}      # scored once, not per threshold
    out = []
    for fp in FPS:
        thr  = torch.quantile(clean[::7], 1 - fp)
        row  = []
        for nm in CURVE_ATTACKS:
            p = att[nm] > thr
            u = (p | GT).sum().item()
            row.append((p & GT).sum().item()/u if u else 1.0)
        out.append(row)
    return np.array(out)                                          # [len(FPS), len(CURVE_ATTACKS)]

with torch.no_grad():
    curves = {"07 gate (energy, 1-sided)": iou_curve(lambda z: one_sided(f_energy(z))),
              "energy, 2-sided":           iou_curve(lambda z: two_sided(f_energy(z))),
              "slope (contrast-free)":     iou_curve(lambda z: two_sided(f_slope(z))),
              "SRM":                       iou_curve(lambda z: two_sided(f_srm(z))),
              "learned fusion (LOIO)":     iou_curve(fuse_score)}

fig, axes = plt.subplots(1, len(CURVE_ATTACKS), figsize=(4.3*len(CURVE_ATTACKS), 4.2), sharey=True)
axes = np.atleast_1d(axes)
for j, (ax, nm) in enumerate(zip(axes, CURVE_ATTACKS)):
    for dn, c in curves.items():
        ax.plot(np.array(FPS)*100, c[:, j], "o-", label=dn)
    ax.axhspan(0.05, 0.16, color="gray", alpha=.18)
    ax.axvline(15, ls="--", c="gray", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("clean false positives (% of pixels)")
    ax.set_title(nm); ax.grid(alpha=.3)
axes[0].set_ylabel("localization IoU"); axes[0].legend(fontsize=8)
plt.suptitle("Better = up and to the LEFT.  Grey band / dashed line = where 07-Attack-Repair landed.")
plt.tight_layout(); plt.show()

i5 = FPS.index(0.05)
print(f"{'detector':26s}" + "".join(f"{n[:11]:>13s}" for n in CURVE_ATTACKS))
print("-"*(26 + 13*len(CURVE_ATTACKS)))
for dn, c in curves.items():
    print(f"{dn:26s}" + "".join(f"{q:13.3f}" for q in c[i5]))
print("(IoU at a 5% clean false-positive rate)")

## 9. The figure asked for: raw image, attacked area, detection

One row per image — the untouched picture, where the attack actually went, the detector's continuous
score, and its thresholded mask with the IoU it achieves. The threshold is the **5%-clean-FP** one
from §8, chosen without ever looking at an attacked image, and each image is scored by the
leave-one-out model that never saw it.

Set `SHOW` to any key of `ZOO` to see a different attack.

In [ ]:
SHOW = REF                      # any key of ZOO: "APGD-T", "Square", "low-freq", ...

with torch.no_grad():
    S   = fuse_score(ZOO[SHOW])
    thr = torch.quantile(fuse_score(raw).flatten().float()[::7], 1 - 0.05)
    P   = S > thr

fig, ax = plt.subplots(N, 4, figsize=(16, 3.9*N)); ax = np.atleast_2d(ax)
for i in range(N):
    gt = masks[i,0].cpu().numpy()
    ax[i,0].imshow(raw[i].permute(1,2,0).cpu())
    ax[i,0].set_title("raw image" if i == 0 else "")
    ax[i,1].imshow(ZOO[SHOW][i].permute(1,2,0).cpu())
    ax[i,1].contour(gt, levels=[.5], colors="lime", linewidths=2)
    ax[i,1].set_title(f"attacked area ({SHOW})" if i == 0 else "")
    ax[i,2].imshow(S[i].cpu(), cmap="inferno")
    ax[i,2].set_title("detector score" if i == 0 else "")
    inter = (P[i] & GT[i]).sum().item(); union = (P[i] | GT[i]).sum().item()
    ax[i,3].imshow(P[i].cpu(), cmap="gray")
    ax[i,3].contour(gt, levels=[.5], colors="lime", linewidths=2)
    ax[i,3].set_title(f"detection @5% FP  (IoU {inter/max(union,1):.2f})")
    for a in ax[i]: a.axis("off")
plt.suptitle(f"Localizing {SHOW} noise - green outline is ground truth, each image scored by a "
             f"model that never saw it", y=1.001)
plt.tight_layout(); plt.show()

## 10. What the perturbation does to the actual obstacle detector

The perturbation was crafted against ResNet-50, so its effect on YOLO is pure **transfer** — which
is the realistic threat model, since an attacker poisoning a scraped dataset does not know the
downstream model. Guarded: if `ultralytics` or its weights are unavailable the notebook carries on.

In [ ]:
def to_np(t):
    return (t.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)

try:
    from ultralytics import YOLO
    yolo = YOLO("yolov8n.pt")
    rows = []
    for i in range(N):
        a = yolo.predict(to_np(raw[i]), verbose=False)[0]
        b = yolo.predict(to_np(ZOO[SHOW][i]), verbose=False)[0]
        rows.append((len(a.boxes), len(b.boxes),
                     a.plot()[..., ::-1].copy(), b.plot()[..., ::-1].copy()))

    print(f"{'image':7s} {'boxes clean':>12s} {'boxes attacked':>15s}")
    for i, (na, nb, _, _) in enumerate(rows):
        print(f"{i:<7d} {na:12d} {nb:15d}")
    ta, tb = sum(r[0] for r in rows), sum(r[1] for r in rows)
    print(f"\ntotal detections: {ta} clean -> {tb} attacked "
          f"({100*(tb-ta)/max(ta,1):+.0f}%), while the attacked region covers "
          f"{masks.mean().item()*100:.0f}% of each image")

    fig, ax = plt.subplots(N, 2, figsize=(9, 4.2*N)); ax = np.atleast_2d(ax)
    for i, (na, nb, ia, ib) in enumerate(rows):
        ax[i,0].imshow(ia); ax[i,0].set_title(f"clean - {na} boxes")
        ax[i,1].imshow(ib); ax[i,1].set_title(f"attacked - {nb} boxes")
        for a in ax[i]: a.axis("off")
    plt.suptitle(f"YOLO on clean vs. {SHOW} (crafted on ResNet-50, so this is transfer)", y=1.0)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("YOLO step unavailable:", type(e).__name__, e)

## Takeaway

1. **The fix for `07`'s negative result is contrast invariance, not a wider filter bank.** Energy
   fails because clean-image energy varies more *between* photos than an attack shifts it *within*
   one. The ratio `e2/e1` measures spectral shape, contrast cancels, and §4 shows its clean median
   barely moves across the 5 images while the energy features move by a large factor.
2. **The score has to be two-sided.** L∞ attacks flatten the local spectrum; smooth L2 attacks
   steepen it. A one-sided detector fitted on L∞ does not merely miss smooth attacks, it ranks them
   *backwards*. `|robust z|` against each image's own median fixes that at almost no cost on the
   L∞ case.
3. **The learned fusion independently rediscovers the same principle** — large opposite-sign
   weights on SRM and plain band energy, which is a ratio, i.e. a scale-invariant spectral-shape
   statistic it was never told to build.
4. **The protocol is what makes the numbers portable.** Leave-one-image-out, fitted on APGD-CE
   only, every other attack reported as transfer, thresholds calibrated on clean images only.
   Nothing here uses an oracle threshold — the thing that inflated a real IoU of ~0.1 into an
   apparent 0.47 in `07`.
5. **The low-frequency attack is still the weak spot** — the hardest column in both tables. That is
   progress over `07`, not a solved problem: a poison hiding in low frequencies is spectrally
   closest to natural image content, which is exactly why it is hard.

### Next

- **Scale up.** Five images with leave-one-out is an honest measurement, not a strong model. The
  fusion has 6 parameters, so a few hundred images will not overfit it; only `N_IMAGES` changes.
- **Replace the linear fusion with a small CNN** over the residual stack. These features are
  hand-designed shape statistics — a conv net can learn the shape basis too.
- **The low-frequency gap is the open problem.** The untested route from `07` still stands: a **VAE
  reconstruction residual**, where encode→decode projects onto Stable Diffusion's learned
  natural-image manifold. That is a learned prior over image content rather than a hand-picked
  band, which is what a low-frequency poison demands.
- **Close the loop** back into [`../05-Noise-Gate/`](../05-Noise-Gate/): swap this score in for the
  HF gate and re-measure quarantine precision and recall on the SD training path.

### References

- Croce & Hein, *Reliable Evaluation of Adversarial Robustness with an Ensemble of Diverse
  Parameter-free Attacks* (AutoAttack), ICML 2020 —
  [arXiv:2003.01690](https://arxiv.org/abs/2003.01690).
- Kim, *Torchattacks: A PyTorch Repository for Adversarial Attacks* —
  [arXiv:2010.01950](https://arxiv.org/abs/2010.01950).
- Liu et al., *Detecting Adversarial Examples Based on Steganalysis* — SRM residuals as
  adversarial-detection features, [arXiv:1806.09186](https://arxiv.org/abs/1806.09186).
- Lorenz et al., *Detecting AutoAttack Perturbations in the Frequency Domain*, ICML 2021 workshop —
  [arXiv:2111.08785](https://arxiv.org/abs/2111.08785). Detects *whether* an image is attacked from
  its spectrum; this notebook asks the harder question of *where*.
- ViT-ReciproCAM — [arXiv:2310.02588](https://arxiv.org/abs/2310.02588), the forward-only,
  batchable scoring idea the detector inherits (see [`../Resources/`](../Resources/)).